In [1]:
import os
import sys
import copy
import importlib
import numpy as np

SCRIPT_DIR = os.path.dirname(os.path.realpath(os.path.join(os.path.dirname(os.getcwd()))))
sys.path.append(SCRIPT_DIR)

import gwfast.gwfastGlobals as glob
import gwfast.waveforms as waveforms
import gwfast.signal as signal
import gwfast.network as network
import gwfast.fisherTools as fTools

LSC Algorithm Library (LAL) is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH
TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


In [25]:
importlib.reload(signal)

<module 'gwfast.signal' from '/Users/yuranzhang/gwfast_AGN/gwfast_moddd/gwfast/signal.py'>

In [18]:
# Two events, with all parameters same except R_orbit
events = {
    'Mc':np.array([50, 50]), 'eta':np.array([0.24, 0.24]), 'dL':np.array([0.8, 0.8]), 'theta':np.array([2.34, 2.34]), 'phi':np.array([5.43, 5.43]), 
    'iota':np.array([0.9*np.pi/2, 0.9*np.pi/2]), 'psi':np.array([1, 1]), 'tGPS':np.array([0, 0]), 'Phicoal':np.array([2.8, 2.8]), 
    'chi1z':np.array([1e-3, 1e-3]), 'chi2z':np.array([1e-3, 1e-3]), 'chi1x':np.array([0, 0]), 'chi2x':np.array([0, 0]), 'chi1y':np.array([0, 0]), 'chi2y':np.array([0, 0]), 
    'LambdaTilde':np.array([0]), 'deltaLambda':np.array([0]), 'ecc':np.array([0]), 
    'phi_L':np.array([0.9*np.pi/2, 0.9*np.pi/2]), 'R_orbit':np.array([200, 300])
}

# Using TaylorF2 for now to speed things up
mywf = waveforms.TaylorF2_RestrictedPN()

In [26]:
# Detector configurations
L1_conf = copy.deepcopy(glob.detectors).pop('L1')
L1_conf['psd_path'] = os.path.join(glob.detPath, 'observing_scenarios_paper', 'AplusDesign.txt')

H1_conf = copy.deepcopy(glob.detectors).pop('H1')
H1_conf['psd_path'] = os.path.join(glob.detPath, 'observing_scenarios_paper', 'AplusDesign.txt')

Virgo_conf = copy.deepcopy(glob.detectors).pop('Virgo')
Virgo_conf['psd_path'] =  os.path.join(glob.detPath, 'observing_scenarios_paper', 'avirgo_O5low_NEW.txt')

# Initialise the GWSignal objects
L1 = signal.GWSignal(mywf, psd_path=L1_conf['psd_path'], 
                        detector_shape=L1_conf['shape'], det_lat=L1_conf['lat'], 
                        det_long=L1_conf['long'], det_xax=L1_conf['xax'],
                        fmin=10
                        )
H1 = signal.GWSignal(mywf, psd_path=H1_conf['psd_path'], 
                        detector_shape=H1_conf['shape'], det_lat=H1_conf['lat'], 
                        det_long=H1_conf['long'], det_xax=H1_conf['xax'],
                        fmin=10
                        )
Virgo = signal.GWSignal(mywf, psd_path=Virgo_conf['psd_path'], 
                        detector_shape=Virgo_conf['shape'], det_lat=Virgo_conf['lat'], 
                        det_long=Virgo_conf['long'], det_xax=Virgo_conf['xax'],
                        fmin=10
                        )

mySignals = {'L1':L1, 'H1':H1, 'Virgo':Virgo}
myNet = network.DetNet(mySignals)

Using ASD from file /Users/yuranzhang/gwfast_AGN/gwfast_moddd/psds/observing_scenarios_paper/AplusDesign.txt 
Initializing jax...
Jax local device count: 8
Jax  device count: 8
Using ASD from file /Users/yuranzhang/gwfast_AGN/gwfast_moddd/psds/observing_scenarios_paper/AplusDesign.txt 
Initializing jax...
Jax local device count: 8
Jax  device count: 8
Using ASD from file /Users/yuranzhang/gwfast_AGN/gwfast_moddd/psds/observing_scenarios_paper/avirgo_O5low_NEW.txt 
Initializing jax...
Jax local device count: 8
Jax  device count: 8


In [27]:
mySNR = myNet.SNR(events, use_lensing=True, return_all=True)
mySNR_nolens = myNet.SNR(events, return_all=True)
print("SNR with lensing:", mySNR)
print("SNR without lensing:", mySNR_nolens)

SNR with lensing: {'L1': Array([14.21029951, 14.35829869], dtype=float64), 'H1': Array([5.99136137, 6.05568808], dtype=float64), 'Virgo': Array([19.13369472, 19.20906588], dtype=float64), 'net': array([24.57493229, 24.73500175])}
SNR without lensing: {'L1': Array([7.2467816, 7.2467816], dtype=float64), 'H1': Array([3.05317012, 3.05317012], dtype=float64), 'Virgo': Array([9.65253476, 9.65253476], dtype=float64), 'net': array([12.4502658, 12.4502658])}


In [28]:
# Fisher matrix (3-detector network) with lensing
FisherMatrsNet = myNet.FisherMatr(events, use_lensing=True)
FisherMatrsNet

Computing Fisher for L1...
Computing derivatives...


Computing Fisher for H1...
Computing derivatives...
Computing Fisher for Virgo...
Computing derivatives...
Done.


array([[[ 2.75391218e+01,  3.29183645e+01],
        [-6.63510120e+03, -7.53675894e+03],
        [-1.26456648e+01, -1.27434313e+01],
        [ 3.92378219e+01,  4.43777826e+01],
        [-4.96276984e+01, -5.33982838e+01],
        [-5.70679084e+00, -5.72189737e+00],
        [-4.56596516e+00, -4.52325808e+00],
        [ 2.88627910e+03,  3.25776039e+03],
        [-1.83462551e+01, -2.01465053e+01],
        [-2.51406302e+03, -2.84185057e+03],
        [-1.56276402e+03, -1.76602269e+03],
        [-2.97441546e-01, -1.78806690e-01],
        [ 3.49709755e-03,  1.37498595e-03]],

       [[-6.63510120e+03, -7.53675894e+03],
        [ 5.38052196e+06,  5.56506863e+06],
        [-9.98102999e+01, -5.47005002e+01],
        [-5.65812225e+04, -5.78435729e+04],
        [ 7.33215061e+04,  7.44464954e+04],
        [ 4.13752568e+03,  4.16353453e+03],
        [ 1.90087398e+04,  1.91139002e+04],
        [-9.72251792e+06, -9.85673224e+06],
        [ 2.81827934e+04,  2.80369237e+04],
        [ 2.50568632e+06,  2.5

In [29]:
# Fisher matrix (3-detector network) without lensing
FisherMatrsNet_nolens = myNet.FisherMatr(events)
FisherMatrsNet_nolens # the two matrices are identical as expected

Computing Fisher for L1...
Computing derivatives...
Computing Fisher for H1...
Computing derivatives...
Computing Fisher for Virgo...
Computing derivatives...
Done.


array([[[ 9.51731309e+00,  9.51731309e+00],
        [-2.08691201e+03, -2.08691201e+03],
        [-3.22935664e+00, -3.22935664e+00],
        [-1.65700428e+01, -1.65700428e+01],
        [-2.03170663e+01, -2.03170663e+01],
        [ 9.10541945e+00,  9.10541945e+00],
        [-1.83892030e+01, -1.83892030e+01],
        [ 8.87491164e+02,  8.87491164e+02],
        [-1.10991445e+01, -1.10991445e+01],
        [-7.83160887e+02, -7.83160887e+02],
        [-4.86459403e+02, -4.86459403e+02]],

       [[-2.08691201e+03, -2.08691201e+03],
        [ 1.43359364e+06,  1.43359364e+06],
        [ 3.14314776e-13,  3.14314776e-13],
        [ 3.04023403e+03,  3.04023403e+03],
        [ 2.26987972e+04,  2.26987972e+04],
        [-5.78157801e+03, -5.78157801e+03],
        [ 1.53470508e+04,  1.53470508e+04],
        [-2.50001245e+06, -2.50001245e+06],
        [ 1.41376225e+04,  1.41376225e+04],
        [ 6.60846431e+05,  6.60846431e+05],
        [ 4.31839977e+05,  4.31839977e+05]],

       [[-3.22935664e+00, -3

In [31]:
Cov, ie = fTools.CovMatr(FisherMatrsNet)
Cov_nolens, ie_nolens = fTools.CovMatr(FisherMatrsNet_nolens)
print("Inversion error with lensing:", ie)
print("Inversion error without lensing:", ie_nolens, "the two values are identical as expected")

Inversion error with lensing: [1.93919803e-07 2.40126928e-06]
Inversion error without lensing: [0.01867676 0.01867676] the two values are identical as expected


In [32]:
print("Covariance matrix for event 1 with lensing:", (np.diag(Cov[:,:,0])))
print("Covariance matrix for event 1 without lensing:", np.sqrt(np.diag(Cov_nolens[:,:,0])))

Covariance matrix for event 1 with lensing: [2.13814537e+01 1.59378618e-01 4.03858345e-03 7.67185843e-04
 1.34366202e-03 1.96725089e-03 9.79759589e-03 2.80511861e-02
 1.04754456e-02 1.51392765e+02 3.44206232e+02 8.77440912e+01
 1.34095893e+04]
Covariance matrix for event 1 without lensing: [3.73419748e+02 1.47548222e+02 4.97976275e+00 6.68042237e-02
 9.96231947e-02 5.65416935e-02 1.30076439e-01 2.42129082e+01
 3.49212471e+04 1.28682647e+03 3.46097109e+03]
